# IssueFix-RL — GRPO Training on Kaggle 2× T4

**Upload this notebook to Kaggle, enable the GPU T4 ×2 accelerator, turn on Internet, attach the two inputs below, then Save & Run All.**

Pipeline:
1. Clone repo + install deps
2. Validate the SFT checkpoint and prompt dataset before any GPU time is spent
3. **Smoke test first** (`SMOKE_TEST = True`): 8 prompts, 128-token completions, a few minutes end to end
4. Full GRPO run — `train.py` spawns one DDP worker per T4
5. Zip the latest LoRA checkpoint → `/kaggle/working/` (Output tab)

---
**Before running:**
- Attach the **SFT checkpoint** (`checkpoint-epoch2-step150`, the model that emits `<think>`/`<answer>`) as a Kaggle Model or Dataset and set `SFT_CHECKPOINT`. The raw Instruct model never emits those tags, so its format reward would sit at 0.
- Attach the prompt data and set `DATA_PATH`. Recommended: upload `datasets/processed/rl/` (built by `datasets/prepare_rl_pool.py`) as a Kaggle dataset and use `rl_train_mix.jsonl`: short function-level problems whose gold solutions pass their own tests. The older `opencode_sft_filtered_sl3072_10000.jsonl` still works but has no tests.
- Add your wandb key as a Kaggle secret named `WANDB_API_KEY` (optional).

**What the reward measures:** response format + Python syntax + similarity to the reference solution. Code is **never executed**, so this run trains format and style, not correctness.

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
from datetime import datetime, timezone

REPO_URL          = "https://github.com/ramprasathk07/IssueFix-RL.git"
REPO_BRANCH       = "feature/grpo-opsd-training"   # switch to "main" once merged
SFT_CHECKPOINT    = "/kaggle/input/issuefix-sft/transformers/qwen0.5-sft/1"  # dir holding config.json + model.safetensors
DATA_PATH         = "/kaggle/input/issuefix-rl-pool/rl_train_mix.jsonl"  # upload datasets/processed/rl/ as a Kaggle dataset
# DATA_PATH       = "/kaggle/input/datasets/ramprasathk07/thinking-traces/opencode_sft_filtered_sl3072_10000.jsonl"  # older prompts, no tests
MAX_PROMPTS       = 800     # worst case ~95 s per 4 x 2048-token rollout → ~800 prompts per 12 h on 2x T4. Rescale from perf/gen_tokens_per_s in the smoke run.
SMOKE_TEST        = True    # first run: 8 prompts, 128 tokens, proves loading/DDP/wandb/saving. Set False for the real run.
RESUME_CHECKPOINT = None    # e.g. "/kaggle/input/my-grpo-ckpt/checkpoint-epoch1-step100"
WANDB_PROJECT     = "issuefix_grpo"
RUN_NAME          = f"qwen0.5_grpo_lora_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, subprocess, sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REPO_DIR = "/kaggle/working/IssueFix-RL"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"Repo: {os.getcwd()} @ {REPO_BRANCH} ({commit})")

In [ ]:
# torch is pre-installed on Kaggle; this installs everything else.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# Kaggle ships torchao 0.10, and current PEFT raises ImportError while wrapping
# LoRA layers whenever an installed torchao is older than 0.16. Nothing in this
# repo uses torchao, so remove it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)

In [ ]:
# Fail early unless Kaggle really assigned two GPUs.
import torch

assert torch.cuda.is_available(), "CUDA is unavailable. Enable the GPU T4 x2 accelerator."
assert torch.cuda.device_count() >= 2, (
    f"GRPO is configured for two GPUs, but Kaggle exposed {torch.cuda.device_count()}. "
    "Select the GPU T4 x2 accelerator."
)
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {props.name}, {props.total_memory / 2**30:.1f} GiB")

In [ ]:
# Pull the wandb key from Kaggle Secrets (Add-ons → Secrets in the notebook editor).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("wandb key loaded from Kaggle Secrets")
except Exception:
    os.environ["WANDB_DISABLED"] = "true"
    print("No WANDB_API_KEY secret found — wandb logging disabled")

In [ ]:
# Validate the SFT checkpoint before any GPU work. GRPO must start from the model
# that already emits <think>/<answer>; RL cannot teach a token that starts at p≈0.
from pathlib import Path
from transformers import AutoTokenizer

ckpt = Path(SFT_CHECKPOINT)
if not (ckpt / "config.json").is_file():
    found = sorted(str(p.parent) for p in Path("/kaggle/input").rglob("config.json"))
    raise FileNotFoundError(
        f"No config.json in {ckpt}. Model directories attached to this notebook:\n  "
        + "\n  ".join(found or ["(none — attach the SFT checkpoint via Add Input)"])
    )
tok = AutoTokenizer.from_pretrained(ckpt)
missing = [t for t in ["<think>", "</think>", "<answer>", "</answer>"] if t not in tok.get_vocab()]
assert not missing, f"{ckpt} is not the SFT checkpoint: its tokenizer lacks {missing}"
weights = sorted(p.name for p in ckpt.iterdir() if p.suffix == ".safetensors")
assert weights, f"No .safetensors weights in {ckpt}"
print(f"SFT checkpoint OK: {ckpt}\n  weights: {weights}\n  vocab: {len(tok):,}")

In [ ]:
import json

data_file = Path(DATA_PATH)
if not data_file.is_file():
    found = sorted(str(p) for p in Path("/kaggle/input").rglob("*.jsonl"))
    raise FileNotFoundError(
        f"Dataset not found: {data_file}. JSONL files attached to this notebook:\n  "
        + "\n  ".join(found or ["(none — attach the data via Add Input)"])
    )
with data_file.open(encoding="utf-8") as handle:
    rows = [json.loads(line) for line in handle if line.strip()]
usable = [
    r for r in rows
    if (r.get("problem") or r.get("prompt")) and (r.get("solution") or r.get("response"))
]
assert usable, "No rows with prompt/problem and solution/response fields"
print(f"Rows: {len(rows):,}  usable: {len(usable):,}  keys: {sorted(rows[0])}")
print("Prompt  :", (usable[0].get("problem") or usable[0].get("prompt"))[:200])
print("Solution:", (usable[0].get("solution") or "")[:200])
del rows, usable

In [ ]:
# Derive a session config; the checked-in configs/grpo.yaml stays unchanged.
import yaml

with open("configs/grpo.yaml", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)
cfg["model_params"]["base_model"] = str(ckpt)
cfg["training_params"]["wandb_project"] = WANDB_PROJECT
cfg["training_params"]["wandb_run_name"] = RUN_NAME
cfg["training_params"]["output_dir"] = "/kaggle/working/outputs/grpo_lora"
cfg["grpo_params"]["max_prompts"] = MAX_PROMPTS
if SMOKE_TEST:
    cfg["grpo_params"]["max_prompts"] = 8
    cfg["grpo_params"]["max_completion_length"] = 128
    # 128 tokens truncates every rollout; keep them in the loss so the smoke run
    # proves a real, nonzero gradient flows through DDP.
    cfg["grpo_params"]["mask_truncated_completions"] = False
    cfg["training_params"]["logging_steps"] = 1
    cfg["training_params"]["save_steps"] = 1000
    cfg["training_params"]["wandb_run_name"] = RUN_NAME + "_smoke"
    cfg["training_params"]["output_dir"] = "/kaggle/working/outputs/grpo_smoke"

runtime_config = Path("/kaggle/working/grpo-runtime.yaml")
with runtime_config.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

tp, gp, dp = cfg["training_params"], cfg["grpo_params"], cfg["dataloader_params"]
prompts_per_step = dp["batch_size"] * tp["num_gpus"] * tp["gradient_accumulation_steps"]
print(yaml.safe_dump({
    "mode": "SMOKE TEST" if SMOKE_TEST else "full run",
    "base_model": cfg["model_params"]["base_model"],
    "prompts": gp["max_prompts"],
    "generations_per_prompt": gp["num_generations"],
    "prompts_per_optimizer_step": prompts_per_step,
    "optimizer_steps": -(-gp["max_prompts"] // prompts_per_step),
    "max_completion_length": gp["max_completion_length"],
    "loss_type": gp["loss_type"],
    "reward_weights": {k: gp[f"{k}_reward_weight"] for k in ("format", "syntax", "reference")},
    "learning_rate": tp["learning_rate"],
    "output_dir": tp["output_dir"],
    "wandb": f"{tp['wandb_project']}/{tp['wandb_run_name']}",
}, sort_keys=False))

In [ ]:
# ── TRAINING ──────────────────────────────────────────────────────────────────
# train.py spawns one DDP worker per T4. Output streams live into this cell and
# the exit code is printed, so a segfault or OOM kill never looks like a stalled bar.
command = [
    sys.executable, "train.py",
    "--method", "grpo",
    "--finetuning", "lora",
    "--config", str(runtime_config),
    "--data", str(data_file),
]
if RESUME_CHECKPOINT:
    command += ["--resume", RESUME_CHECKPOINT]
print("Launching:", " ".join(command), flush=True)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TQDM_MININTERVAL"] = "30"  # the progress bar redraws as new lines through a pipe
proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
print("exit code:", code)
if code != 0:
    # Negative codes are signals: -6 abort, -9 host OOM kill, -11 segfault.
    raise SystemExit(f"GRPO training failed with exit code {code}; see the log above.")

In [ ]:
# ── OUTPUTS ───────────────────────────────────────────────────────────────────
# List checkpoints and zip the latest so it shows up in the notebook Output tab.
import shutil

out_dir = Path(cfg["training_params"]["output_dir"])
checkpoints = sorted(out_dir.glob("checkpoint-*"), key=lambda p: p.stat().st_mtime)
assert checkpoints, f"No checkpoints under {out_dir}; check the training log above"
for checkpoint in checkpoints:
    size_mb = sum(f.stat().st_size for f in checkpoint.rglob("*") if f.is_file()) / 2**20
    print(f"{checkpoint.name}: {size_mb:.1f} MiB")
latest = checkpoints[-1]
archive = shutil.make_archive(
    str(Path("/kaggle/working") / f"{tp['wandb_run_name']}-{latest.name}"), "zip", root_dir=latest
)
print("Archived:", archive)

## Reading the wandb run

Metrics land in wandb sections by prefix, averaged over each logging window (peak metrics use the max).

| Plot | Healthy | If not |
|---|---|---|
| `reward/format` | Starts low (~5% of samples at T=0.7 in local calibration) and climbs | Flat near 0 → the format signal is too sparse to learn from |
| `reward/frac_zero_std_groups` | Well below 0.6 | Most prompts give no learning signal |
| `kl/ref_k3`, `kl/ref_k1` | Small, slow rise | Fast climb → the policy is leaving the SFT model; lower LR or set `beta > 0` |
| `policy/entropy` | Slow decline | Collapse toward 0 → mode collapse; a sharp rise → instability |
| `loss/policy`, `loss/kl_penalty` | Policy term noisy around 0 | `loss/kl_penalty` stays 0 while `beta = 0` |
| `completion/truncated_fraction`, `completion/length_mean` | Low truncation, stable length | Length rising with reward while `reward/syntax` is flat → length gaming |
| `perf/gen_tokens_per_s`, `perf/gpu_mem_peak_gb` | Steady | Use the smoke run's tokens/s to size `MAX_PROMPTS` |
| `samples/completions` table | Readable `<think>…</think><answer>…</answer>` | Spot-check it; reward going up is never proof on its own |

`kl/old_policy`, `policy/clip_fraction`, and `policy/ratio_mean` sit at ~0, ~0, and 1 by construction: each rollout gets one policy update, so its ratio against the rollout policy is 1.

**Resume in a later session:** attach the checkpoint directory as a dataset, set `RESUME_CHECKPOINT`, and keep `SFT_CHECKPOINT` unchanged; the LoRA adapter sits on top of it.